In [47]:
import pandas as pd
import zipfile
from pathlib import Path
import geopandas as gpd
import numpy as np
import os



In [23]:
df = pd.read_csv("opensecrets_dark_money.csv")

Clean

In [33]:
money_cols = ["Total", "For Dems", "Against Dems", "For Repubs", "Against Repubs"]

for col in money_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace("-", "0")
        .replace("", "0")
    )

    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

df["has_dark_money_data"] = True

Keep only House Districts

In [34]:
df_house = df[
    df["race_id"]
    .astype(str)
    .str.upper()
    .str.match(r"^[A-Z]{2}\d{2}$", na=False)
].copy()

df_house["race_id"] = df_house["race_id"].astype(str).str.upper()
df_house["state_abbr"] = df_house["race_id"].str[:2]
df_house["district"] = df_house["race_id"].str[2:]

# Marker: these are rows that actually appeared in your scraped dark-money data
df_house["has_dark_money_data"] = True

print(f"Total scraped rows: {len(df)}")
print(f"House district rows: {len(df_house)}")
print(df_house[["race_id", "state_abbr", "district", "has_dark_money_data"]].head())

Total scraped rows: 99
House district rows: 99
  race_id state_abbr district  has_dark_money_data
0    OH13         OH       13                 True
1    OH09         OH       09                 True
2    NY22         NY       22                 True
3    MI07         MI       07                 True
4    NM02         NM       02                 True


In [35]:
df_house.to_csv("opensecrets_dark_money.csv", index=False)

Open District Zip

In [27]:
# outer bundle zip
outer_zip = Path("cb_2024_us_all_500k.zip")

# folder where inner zips will be extracted
extract_dir = Path("census_2024_files")
extract_dir.mkdir(exist_ok=True)

# extract only the congressional district zip
with zipfile.ZipFile(outer_zip, "r") as z:
    z.extract("cb_2024_us_cd119_500k.zip", extract_dir)

# read the actual congressional district shapefile zip
cd_zip = extract_dir / "cb_2024_us_cd119_500k.zip"

cd = gpd.read_file(cd_zip)

print(cd.head())
print(cd.columns)
print(cd.crs)
print(len(cd))

  STATEFP CD119FP        GEOIDFQ GEOID                   NAMELSAD LSAD  \
0      06      50  5001900US0650  0650  Congressional District 50   C2   
1      17      07  5001900US1707  1707   Congressional District 7   C2   
2      06      39  5001900US0639  0639  Congressional District 39   C2   
3      36      06  5001900US3606  3606   Congressional District 6   C2   
4      39      11  5001900US3911  3911  Congressional District 11   C2   

  CDSESSN      ALAND      AWATER  \
0     119  664411673   402346282   
1     119  177223764     2189439   
2     119  735864387     4211110   
3     119   66826735      796953   
4     119  558382153  2030656323   

                                            geometry  
0  POLYGON ((-117.28217 32.83955, -117.28117 32.8...  
1  POLYGON ((-87.92061 41.8827, -87.913 41.88209,...  
2  POLYGON ((-117.54969 33.95409, -117.54891 33.9...  
3  POLYGON ((-73.91557 40.74458, -73.91523 40.746...  
4  POLYGON ((-81.87891 41.39712, -81.87735 41.399...  
Index(['

Create same race_ids for join

In [36]:
state_fips_to_abbr = {
    "01": "AL", "02": "AK", "04": "AZ", "05": "AR", "06": "CA",
    "08": "CO", "09": "CT", "10": "DE", "11": "DC", "12": "FL",
    "13": "GA", "15": "HI", "16": "ID", "17": "IL", "18": "IN",
    "19": "IA", "20": "KS", "21": "KY", "22": "LA", "23": "ME",
    "24": "MD", "25": "MA", "26": "MI", "27": "MN", "28": "MS",
    "29": "MO", "30": "MT", "31": "NE", "32": "NV", "33": "NH",
    "34": "NJ", "35": "NM", "36": "NY", "37": "NC", "38": "ND",
    "39": "OH", "40": "OK", "41": "OR", "42": "PA", "44": "RI",
    "45": "SC", "46": "SD", "47": "TN", "48": "TX", "49": "UT",
    "50": "VT", "51": "VA", "53": "WA", "54": "WV", "55": "WI",
    "56": "WY",
}

cd["state_abbr"] = cd["STATEFP"].map(state_fips_to_abbr)
cd["district"] = cd["CD119FP"]
cd["race_id"] = cd["state_abbr"] + cd["district"]

print(cd[["NAMELSAD", "STATEFP", "CD119FP", "race_id"]].head())

                    NAMELSAD STATEFP CD119FP race_id
0  Congressional District 50      06      50    CA50
1   Congressional District 7      17      07    IL07
2  Congressional District 39      06      39    CA39
3   Congressional District 6      36      06    NY06
4  Congressional District 11      39      11    OH11


In [ ]:

money_cols = ["Total", "For Dems", "Against Dems", "For Repubs", "Against Repubs"]


df_house_join = df_house[
    [
        "race_id",
        "Race",
        "Total",
        "For Dems",
        "Against Dems",
        "For Repubs",
        "Against Repubs",
    ]
].copy()

# Marker: district appeared in scraped dark-money data
df_house_join["has_dark_money_data"] = True

# Joining opensecrets data to maps
map_gdf = cd.merge(
    df_house_join,
    on="race_id",
    how="left"
)

# removing Alaska and Hawaii to not distort the map
map_gdf = map_gdf[
    map_gdf["state_abbr"].notna()
    & ~map_gdf["state_abbr"].isin(["AK", "HI"])
].copy()


# creating datawrapper coloring colum so that 0 is shown as grey (no data) but not colored as $0
map_gdf["Total_for_map_color"] = np.where(
    map_gdf["has_dark_money_data"].eq(True) & (map_gdf["Total"] > 0),
    map_gdf["Total"],
    np.nan
)


# display columns
for col in money_cols:
    map_gdf[f"{col}_display"] = map_gdf[col].fillna(0)


map_gdf["Race"] = map_gdf["Race"].fillna(
    map_gdf["race_id"] + " — no OpenSecrets row"
)


print(map_gdf[
    [
        "race_id",
        "NAMELSAD",
        "Race",
        "Total",
        "Total_for_map_color",
        "Total_display",
        "has_dark_money_data"
    ]
].head(20))

print(f"Rows after removing territories, AK, HI: {len(map_gdf)}")
print(sorted(map_gdf["state_abbr"].dropna().unique()))

print("Dark-money districts colored:", map_gdf["Total_for_map_color"].notna().sum())
print("Grey / no-data districts:", map_gdf["Total_for_map_color"].isna().sum())

   race_id                   NAMELSAD                       Race     Total  \
0     CA50  Congressional District 50  CA50 — no OpenSecrets row       NaN   
1     IL07   Congressional District 7        Illinois District 7   43269.0   
2     CA39  Congressional District 39  CA39 — no OpenSecrets row       NaN   
3     NY06   Congressional District 6  NY06 — no OpenSecrets row       NaN   
4     OH11  Congressional District 11  OH11 — no OpenSecrets row       NaN   
5     NJ03   Congressional District 3  NJ03 — no OpenSecrets row       NaN   
6     SC01   Congressional District 1  SC01 — no OpenSecrets row       NaN   
7     WI05   Congressional District 5       Wisconsin District 5   14200.0   
8     CA36  Congressional District 36  CA36 — no OpenSecrets row       NaN   
9     NY14  Congressional District 14  NY14 — no OpenSecrets row       NaN   
10    TN06   Congressional District 6  TN06 — no OpenSecrets row       NaN   
11    CA21  Congressional District 21  CA21 — no OpenSecrets row

## Export Full Map

In [ ]:

# this holds all dark money districts and all geometry
# needs to be as simple as possible to not hit DW file size limits
dw_focus_only = map_gdf.copy()


geojson_file = "datawrapper_all_dark_money_districts_geo.geojson"

geo_cols = [
    "race_id",
    "state_abbr",
    "NAMELSAD",
    "geometry"
]

map_small = dw_focus_only[geo_cols].copy()

# DW CRS
map_small = map_small.to_crs(epsg=4326)

map_small["geometry"] = map_small["geometry"].simplify(
    tolerance=0.01,
    preserve_topology=True
)

map_small.to_file(
    geojson_file,
    driver="GeoJSON"
)


csv_file = "datawrapper_all_dark_money_district_csv.csv"

dw_csv = dw_focus_only.copy()

dw_csv = dw_csv.rename(columns={
    "NAMELSAD": "district_name",
    "Race": "race_label",
    "Total_for_map_color": "total_for_map_color",
    "Total_display": "total_display",
    "For Dems_display": "for_dems_display",
    "Against Dems_display": "against_dems_display",
    "For Repubs_display": "for_repubs_display",
    "Against Repubs_display": "against_repubs_display",
})

csv_cols = [
    "race_id",
    "state_abbr",
    "district_name",
    "race_label",
    "has_dark_money_data",
    "total_for_map_color",
    "total_display",
    "for_dems_display",
    "against_dems_display",
    "for_repubs_display",
    "against_repubs_display"
]

dw_csv[csv_cols].to_csv(
    csv_file,
    index=False
)

print("Saved CSV:", csv_file)
print(dw_csv[csv_cols].head())

print("Saved:")
print(geojson_file)
print(csv_file)

print("GeoJSON rows:", len(map_small))
print("CSV rows:", len(dw_focus_only))

print("GeoJSON size MB:", round(os.path.getsize(geojson_file) / 1024 / 1024, 2))
print("CSV size MB:", round(os.path.getsize(csv_file) / 1024 / 1024, 2))

print("Colored districts:", dw_focus_only["Total_for_map_color"].notna().sum())
print("Grey / no-data districts:", dw_focus_only["Total_for_map_color"].isna().sum())

print(dw_focus_only[
    [
        "race_id",
        "NAMELSAD",
        "Race",
        "has_dark_money_data",
        "Total_for_map_color",
        "Total_display"
    ]
].head(20))

Saved CSV: datawrapper_all_dark_money_district_csv.csv
  race_id state_abbr              district_name                 race_label  \
0    CA50         CA  Congressional District 50  CA50 — no OpenSecrets row   
1    IL07         IL   Congressional District 7        Illinois District 7   
2    CA39         CA  Congressional District 39  CA39 — no OpenSecrets row   
3    NY06         NY   Congressional District 6  NY06 — no OpenSecrets row   
4    OH11         OH  Congressional District 11  OH11 — no OpenSecrets row   

  has_dark_money_data  total_for_map_color  total_display  for_dems_display  \
0                 NaN                  NaN            0.0               0.0   
1                True              43269.0        43269.0           43269.0   
2                 NaN                  NaN            0.0               0.0   
3                 NaN                  NaN            0.0               0.0   
4                 NaN                  NaN            0.0               0.0   

 

## Get Competitive Districts

In [ ]:
# all competitive districts according to Cook Political Report as of October 2024
competitive_ratings = {
    "CA49": "Leans Dem",
    "CT05": "Leans Dem",
    "MD06": "Leans Dem",
    "NM02": "Leans Dem",
    "NY22": "Leans Dem",
    "OH13": "Leans Dem",
    "TX34": "Leans Dem",

    "CA13": "Tilt Dem",
    "IA01": "Tilt Dem",
    "MI08": "Tilt Dem",
    "NC01": "Tilt Dem",
    "NE02": "Tilt Dem",
    "NY04": "Tilt Dem",
    "NY19": "Tilt Dem",
    "OH09": "Tilt Dem",
    "OR05": "Tilt Dem",
    "PA07": "Tilt Dem",
    "PA08": "Tilt Dem",
    "VA07": "Tilt Dem",

    "AZ06": "Toss-up",
    "CA22": "Toss-up",
    "CA27": "Toss-up",
    "CA45": "Toss-up",
    "CA47": "Toss-up",
    "CO08": "Toss-up",
    "ME02": "Toss-up",
    "WA03": "Toss-up",

    "AK00": "Tilt Rep",
    "AZ01": "Tilt Rep",
    "CA41": "Tilt Rep",
    "IA03": "Tilt Rep",
    "MI07": "Tilt Rep",
    "NJ07": "Tilt Rep",
    "NY17": "Tilt Rep",
    "PA10": "Tilt Rep",
    "VA02": "Tilt Rep",
    "WI03": "Tilt Rep",

    "CO03": "Leans Rep",
    "MI10": "Leans Rep",
    "MT01": "Leans Rep",
}

In [ ]:
# Competitive overlay map:


required_cols = [
    "race_id",
    "state_abbr",
    "NAMELSAD",
    "Race",
    "Total",
    "Total_for_map_color",
    "Total_display",
    "For Dems_display",
    "Against Dems_display",
    "For Repubs_display",
    "Against Repubs_display",
    "has_dark_money_data",
    "geometry"
]

missing_cols = [col for col in required_cols if col not in map_gdf.columns]

if missing_cols:
    raise KeyError(f"Missing columns in map_gdf: {missing_cols}")

map_competitive_overlay_gdf = map_gdf.copy()


map_competitive_overlay_gdf["competitive_rating"] = (
    map_competitive_overlay_gdf["race_id"].map(competitive_ratings)
)

map_competitive_overlay_gdf["is_competitive"] = (
    map_competitive_overlay_gdf["competitive_rating"].notna()
)


map_competitive_overlay_gdf["has_dark_money"] = (
    map_competitive_overlay_gdf["has_dark_money_data"].eq(True)
    & map_competitive_overlay_gdf["Total"].gt(0)
)


map_competitive_overlay_gdf["dark_money_and_competitive"] = (
    map_competitive_overlay_gdf["has_dark_money"]
    & map_competitive_overlay_gdf["is_competitive"]
)


map_competitive_overlay_gdf["competitive_rating_display"] = pd.NA

map_competitive_overlay_gdf.loc[
    map_competitive_overlay_gdf["dark_money_and_competitive"],
    "competitive_rating_display"
] = map_competitive_overlay_gdf["competitive_rating"]

map_competitive_overlay_gdf.loc[
    map_competitive_overlay_gdf["has_dark_money"]
    & ~map_competitive_overlay_gdf["is_competitive"],
    "competitive_rating_display"
] = "Not rated competitive"


map_competitive_overlay_gdf["dark_money_competitive_category"] = pd.NA

map_competitive_overlay_gdf.loc[
    map_competitive_overlay_gdf["has_dark_money"]
    & ~map_competitive_overlay_gdf["is_competitive"],
    "dark_money_competitive_category"
] = "Dark money only"

map_competitive_overlay_gdf.loc[
    map_competitive_overlay_gdf["dark_money_and_competitive"],
    "dark_money_competitive_category"
] = "Dark money + competitive"


map_competitive_overlay_gdf["competitive_pattern"] = pd.NA

map_competitive_overlay_gdf.loc[
    map_competitive_overlay_gdf["dark_money_and_competitive"],
    "competitive_pattern"
] = "Competitive dark-money district"

map_competitive_overlay_gdf["dark_money_color"] = (
    map_competitive_overlay_gdf["Total_for_map_color"]
)


print("Rows:", len(map_competitive_overlay_gdf))
print("Dark-money districts:", map_competitive_overlay_gdf["has_dark_money"].sum())
print("Competitive districts overall:", map_competitive_overlay_gdf["is_competitive"].sum())
print("Dark money + competitive:", map_competitive_overlay_gdf["dark_money_and_competitive"].sum())
print("Patterned districts:", map_competitive_overlay_gdf["competitive_pattern"].notna().sum())

print(
    map_competitive_overlay_gdf.loc[
        map_competitive_overlay_gdf["competitive_pattern"].notna(),
        [
            "race_id",
            "NAMELSAD",
            "Total",
            "competitive_rating",
            "competitive_rating_display",
            "dark_money_competitive_category",
            "competitive_pattern"
        ]
    ].sort_values("Total", ascending=False)
)

Rows: 433
Dark-money districts: 98
Competitive districts overall: 39
Dark money + competitive: 29
Patterned districts: 29
    race_id                   NAMELSAD     Total competitive_rating  \
301    OH13  Congressional District 13  412584.0          Leans Dem   
438    OH09   Congressional District 9  399740.0           Tilt Dem   
17     NY22  Congressional District 22  398660.0          Leans Dem   
346    MI07   Congressional District 7  393552.0           Tilt Rep   
69     NM02   Congressional District 2  369701.0          Leans Dem   
209    PA07   Congressional District 7  271811.0           Tilt Dem   
51     CA47  Congressional District 47  160742.0            Toss-up   
368    CA45  Congressional District 45  114440.0            Toss-up   
116    CO08   Congressional District 8   89604.0            Toss-up   
155    CA49  Congressional District 49   71199.0          Leans Dem   
228    IA03   Congressional District 3   58033.0           Tilt Rep   
113    CA41  Congressional

In [ ]:
# exporting to GEOJSON and csv for copy of Map A

geojson_file_overlay = "datawrapper_map_competitive_overlay_geo.geojson"
csv_file_overlay = "datawrapper_map_competitive_overlay_csv.csv"


geo_cols_overlay = [
    "race_id",
    "state_abbr",
    "NAMELSAD",
    "geometry"
]

map_competitive_overlay_small = map_competitive_overlay_gdf[geo_cols_overlay].copy()
map_competitive_overlay_small = map_competitive_overlay_small.to_crs(epsg=4326)

map_competitive_overlay_small["geometry"] = (
    map_competitive_overlay_small["geometry"].simplify(
        tolerance=0.01,
        preserve_topology=True
    )
)

map_competitive_overlay_small.to_file(
    geojson_file_overlay,
    driver="GeoJSON"
)


map_competitive_overlay_csv = map_competitive_overlay_gdf.copy()

map_competitive_overlay_csv = map_competitive_overlay_csv.rename(columns={
    "NAMELSAD": "district_name",
    "Race": "race_label",
    "Total_display": "total_display",
    "For Dems_display": "for_dems_display",
    "Against Dems_display": "against_dems_display",
    "For Repubs_display": "for_repubs_display",
    "Against Repubs_display": "against_repubs_display",
})

csv_cols_overlay = [
    "race_id",
    "state_abbr",
    "district_name",
    "race_label",

    # competitive columns
    "is_competitive",
    "competitive_rating",
    "competitive_rating_display",
    "competitive_pattern",

    # dark-money columns
    "has_dark_money",
    "dark_money_competitive_category",
    "dark_money_color",

    # tooltip values for DW
    "total_display",
    "for_dems_display",
    "against_dems_display",
    "for_repubs_display",
    "against_repubs_display",
]

map_competitive_overlay_csv[csv_cols_overlay].to_csv(
    csv_file_overlay,
    index=False
)




print("Dark-money districts:", map_competitive_overlay_gdf["has_dark_money"].sum())
print(
    "Dark money + competitive:",
    (map_competitive_overlay_gdf["dark_money_competitive_category"] == "Dark money + competitive").sum()
)

Saved competitive overlay files:
datawrapper_map_competitive_overlay_geo.geojson
datawrapper_map_competitive_overlay_csv.csv
GeoJSON rows: 433
CSV rows: 433
GeoJSON size MB: 1.15
CSV size MB: 0.05
Patterned competitive districts: 29
Dark-money districts: 98
Dark money + competitive: 29
